# Abstract decoder
What does a small byte-level language model learn?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.model import ByteTokenizer, Decoder, DecoderConfig
config = DecoderConfig(width=32, layers=1, context=64) if smoke else DecoderConfig()
model = Decoder(config)
tokens = torch.tensor([ByteTokenizer.encode('The trial measured recovery.')])
print('parameters:', sum(p.numel() for p in model.parameters()))
print('logits:', tuple(model(tokens).shape))
print('tied weights:', model.embedding.weight is model.output.weight)

In [ ]:
from pythonic.data import fixtures, load_papers, split_papers
from pythonic.training import TrainConfig, train_decoder
import matplotlib.pyplot as plt
if smoke:
    training = validation = fixtures()
else:
    split = split_papers(load_papers(root / 'data/pubmedqa.json'))
    training, validation = split.train, split.validation
run = TrainConfig(steps=3 if smoke else 20, accumulation=1)
model, report = train_decoder(training, validation, config, run, root / 'artifacts/notebook-decoder')
plt.plot(report['train_loss'])
plt.xlabel('Optimizer step')
plt.ylabel('Training cross-entropy')
print('validation cross-entropy:', report['validation_loss'])

Short runs test the pipeline; they do not establish language quality. Byte-level loss is not comparable to subword-token loss.